In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The function `range` takes two arguments:
- `start` is an integer,
- `stop` is an integer.

It returns the list $[\texttt{start}, \cdots, \texttt{stop}]$.

In [ ]:
let range (start : int) (stop : int) : int list =
  List.init (max 0 (stop - start + 1)) (fun i -> start + i)

In [ ]:
import_notebook "02-Backtracking-Constraint-Solver.ipynb"

# Saving the Goat

An agricultural economist has to sell a *wolf*, a *goat*, and a *cabbage* on a market place.  In order to
reach the market place, she has to cross a river.  The boat that she can use is so small that it can
only accommodate either the goat, the wolf, or the cabbage in addition to the agricultural economist.
Now if the agricultural economist leaves the wolf alone with the goat, the wolf will eat the goat.
If, instead, the agricultural economist leaves the goat with the cabbage, the goat will eat the cabbage.
Is it possible for the agricultural economist to develop a schedule that allows her to cross the river
without either the goat or the cabbage being eaten?

We will encode this problem as a *symbolic transition system* and then solve it with the help of our *constraint solver*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossing of the river.  We use the following variables:
* $\texttt{f}i$ for $i\in\{0, \cdots, n\}$ is the number of farmers on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{w}i$ for $i\in\{0, \cdots, n\}$ is the number of wolves on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{g}i$ for $i\in\{0, \cdots, n\}$ is the number of goats on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{c}i$ for $i\in\{0, \cdots, n\}$ is the number of cabbages on the western shore after the 
  $i^{\textrm{th}}$ crossing.

As the variables of a CSP have to start with a lower case letter, we write them as `f`$i$, `w`$i$, `g`$i$, and `c`$i$.

## Auxiliary Functions

The following functions generate the *formulas* required by the constraint solver.  A formula is a string containing an <samp>OCaml</samp> expression of type `bool`.

The invariant guarantees that there is no problem on either shore of the river. There is no problem if on the shore where there is no farmer, the following conditions hold:
* If the wolf is on that shore, the goat must be on the opposite shore.
* If the goat is on that shore, the cabbage has to be on the opposite shore.

The function `invariant` takes four arguments:
- `f`, `w`, `g`, and `c` are the variables for the number of farmers, wolves, goats, and cabbages on the western shore.

It returns the invariant as a formula.

Replace every occurrence of `failwith "your code here"` with your code.  The function `failwith` raises an exception.  As it can be used in place of an expression of any type, it serves as a placeholder.

In [ ]:
let invariant (f : string) (w : string) (g : string) (c : string) : string =
  failwith "your code here"

The transition formula defines a valid river crossing:
* The farmer travels in every crossing.
* She takes at most one of her goods with her in the boat.
* If the farmer travels from the western shore to the eastern shore, the number of goods on the western shore have to decrease, otherwise they have to increase.

The function `transition` takes eight arguments:
- `f_a`, `w_a`, `g_a`, and `c_a` are the variables describing the state before the crossing,
- `f_b`, `w_b`, `g_b`, and `c_b` are the variables describing the state after the crossing.

It returns the formula describing the transition.  In <samp>OCaml</samp>, the expression `if b then e1 else e2` can be used inside of other expressions, like the conditional operator `b ? e1 : e2` in <samp>C</samp> and the conditional expression `e1 if b else e2` in <samp>Python</samp>.

In [ ]:
let transition (f_a : string) (w_a : string) (g_a : string) (c_a : string)
               (f_b : string) (w_b : string) (g_b : string) (c_b : string) : string =
  (* Create a list west_to_east of the formulas that have to hold for a crossing from
     west to east, a list east_to_west of the formulas for a crossing from east to west,
     and combine them with the formula stating that the farmer crosses the river. *)
  failwith "your code here"

The function `wgc_csp` takes one argument:
- `n` is the number of crossings.

It creates a CSP that tries to solve the problem with `n` crossings.

In [ ]:
let wgc_csp (n : int) : csp =
  let vars    : string list = failwith "your code here" in
  let values  : int list    = failwith "your code here" in
  let constrs : string list =
    [failwith "your code here"]     (* Start *)
    @ [failwith "your code here"]   (* Goal *)
    @ failwith "your code here"     (* Invariant *)
    @ failwith "your code here"     (* Transition Relation *)
  in
  { variables = vars; values = values; constraints = constrs }

In [ ]:
wgc_csp 3

The function `find_solution` takes the argument `()`.  It computes a solution to the problem by iteratively increasing the number of crossings `n`.  The result is the pair `(n, solution)`.

In [ ]:
let find_solution () : int * assignment =
  let rec try_crossings n =
    Printf.printf "Checking n = %d...\n" n;
    match solve (wgc_csp n) with
    | Some solution -> (n, solution)
    | None          -> try_crossings (n + 2)
  in
  try_crossings 1

In [ ]:
let (n, solution) = time "find_solution" find_solution

## Visualizing the Result

The function `repeat` takes two arguments:
- `s` is a string,
- `k` is a natural number.

It returns the string that consists of `k` copies of `s`.

The function `show_solution` takes two arguments:
- `solution` is a solution of the CSP,
- `n` is the number of crossings.

It prints the states of the solution and the passengers of the boat for every crossing.

In [ ]:
let repeat (s : string) (k : int) : string =
  String.concat "" (List.init k (fun _ -> s))

let show_solution (solution : assignment) (n : int) : unit =
  let get x i = RMap.find (Printf.sprintf "%s%d" x i) solution in
  for i = 0 to n do
    let f = get "f" i and w = get "w" i and g = get "g" i and c = get "c" i in
    let left_side  = repeat "🧑‍🌾" f ^ repeat "🐺" w ^ repeat "🐐" g ^ repeat "🥦" c in
    let right_side = repeat "🧑‍🌾" (1 - f) ^ repeat "🐺" (1 - w) ^ repeat "🐐" (1 - g) ^ repeat "🥦" (1 - c) in
    let gap        = String.make (28 - (f + w + g + c) * 2) ' ' in
    print_endline (left_side ^ gap ^ right_side);
    if i < n then begin
      let next_w = get "w" (i + 1) and next_g = get "g" (i + 1) and next_c = get "c" (i + 1) in
      if f = 1 then
        Printf.printf "%s>>>  🧑‍🌾%s%s%s  >>>\n" (String.make 10 ' ')
          (repeat "🐺" (w - next_w)) (repeat "🐐" (g - next_g)) (repeat "🥦" (c - next_c))
      else
        Printf.printf "%s<<<  🧑‍🌾%s%s%s  <<<\n" (String.make 10 ' ')
          (repeat "🐺" (next_w - w)) (repeat "🐐" (next_g - g)) (repeat "🥦" (next_c - c))
    end
  done

In [ ]:
show_solution solution n

The function `show_animated_solution` takes two arguments:
- `solution` is a solution of the CSP,
- `n` is the number of crossings.

It displays an animation of the solution.  As in the notebook `05-Missionaries-STS.ipynb`, the HTML code is stored in the quoted string `template`, which contains the placeholders `__UID__`, `__N__`, and `__STATES__`.  The function `replace_all s pattern by` replaces every occurrence of the string `pattern` in `s` by the string `by`.

In [ ]:
#require "jupyter.notebook";;

In [ ]:
let replace_all (s : string) (pattern : string) (by : string) : string =
  let n = String.length s and k = String.length pattern in
  let buffer = Buffer.create n in
  let rec loop i =
    if i > n - k then
      Buffer.add_string buffer (String.sub s i (n - i))
    else if String.sub s i k = pattern then begin
      Buffer.add_string buffer by;
      loop (i + k)
    end else begin
      Buffer.add_char buffer s.[i];
      loop (i + 1)
    end
  in
  loop 0;
  Buffer.contents buffer

let template = {|
    <div style="font-family: sans-serif; max-width: 600px; margin: 0 auto; text-align: center;">
        <div style="margin-bottom: 15px;">
            <button id="btn-reset-__UID__" style="padding: 8px 16px; cursor: pointer;">Reset</button>
            <button id="btn-prev-__UID__" style="padding: 8px 16px; cursor: pointer;">Previous Step</button>
            <button id="btn-next-__UID__" style="padding: 8px 16px; cursor: pointer; font-weight: bold;">Next Step</button>
            <span id="label-step-__UID__" style="margin-left: 15px; font-weight: bold; font-size: 16px;">Step 0 of __N__</span>
        </div>
        
        <svg width="600" height="250" style="background-color: #e0f7fa; border: 1px solid #b2ebf2; border-radius: 8px;">
            <rect x="0" y="0" width="150" height="250" fill="#a5d6a7" />
            <rect x="450" y="0" width="150" height="250" fill="#a5d6a7" />
            
            <text x="75" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Western Shore</text>
            <text x="525" y="30" font-size="14" font-weight="bold" fill="#2e7d32" text-anchor="middle">Eastern Shore</text>

            <text id="west-f-__UID__" x="75" y="80" font-size="28" text-anchor="middle"></text>
            <text id="west-w-__UID__" x="75" y="120" font-size="28" text-anchor="middle"></text>
            <text id="west-g-__UID__" x="75" y="160" font-size="28" text-anchor="middle"></text>
            <text id="west-c-__UID__" x="75" y="200" font-size="28" text-anchor="middle"></text>
            
            <text id="east-f-__UID__" x="525" y="80" font-size="28" text-anchor="middle"></text>
            <text id="east-w-__UID__" x="525" y="120" font-size="28" text-anchor="middle"></text>
            <text id="east-g-__UID__" x="525" y="160" font-size="28" text-anchor="middle"></text>
            <text id="east-c-__UID__" x="525" y="200" font-size="28" text-anchor="middle"></text>

            <g id="boat-__UID__" style="transition: transform 1.5s ease-in-out;">
                <rect x="160" y="160" width="80" height="40" rx="8" fill="#8d6e63" stroke="#5d4037" stroke-width="2"/>
                <text id="boat-p-__UID__" x="200" y="188" font-size="22" text-anchor="middle"></text>
            </g>
        </svg>
    </div>

    <script>
    (function() {
        const states = __STATES__;
        let currentStep = 0;
        let isAnimating = false;
        
        const boat = document.getElementById('boat-__UID__');
        const boatP = document.getElementById('boat-p-__UID__');
        const stepLabel = document.getElementById('label-step-__UID__');
        
        const elements = {
            west: {
                F: document.getElementById('west-f-__UID__'),
                W: document.getElementById('west-w-__UID__'),
                G: document.getElementById('west-g-__UID__'),
                C: document.getElementById('west-c-__UID__')
            },
            east: {
                F: document.getElementById('east-f-__UID__'),
                W: document.getElementById('east-w-__UID__'),
                G: document.getElementById('east-g-__UID__'),
                C: document.getElementById('east-c-__UID__')
            }
        };
        
        const icons = { F: '🧑‍🌾', W: '🐺', G: '🐐', C: '🥦' };
        
        // Web Audio API context for the motor sound
        let audioCtx;
        function playMotorSound() {
            if (!audioCtx) audioCtx = new (window.AudioContext || window.webkitAudioContext)();
            if (audioCtx.state === 'suspended') audioCtx.resume();
            
            const osc = audioCtx.createOscillator();
            const filter = audioCtx.createBiquadFilter();
            const gain = audioCtx.createGain();
            
            osc.type = 'sawtooth';
            osc.frequency.value = 12; 
            filter.type = 'lowpass';
            filter.frequency.value = 150;

            gain.gain.setValueAtTime(0, audioCtx.currentTime);
            gain.gain.linearRampToValueAtTime(0.6, audioCtx.currentTime + 0.2); 
            
            osc.connect(filter);
            filter.connect(gain);
            gain.connect(audioCtx.destination);
            osc.start();
            
            return { osc, gain };
        }
        
        function stopMotorSound(sound) {
            if (sound) {
                sound.gain.gain.linearRampToValueAtTime(0, audioCtx.currentTime + 0.4);
                setTimeout(() => sound.osc.stop(), 400);
            }
        }

        function renderStaticState(step) {
            const s = states[step];
            elements.west.F.textContent = icons.F.repeat(s.F);
            elements.west.W.textContent = icons.W.repeat(s.W);
            elements.west.G.textContent = icons.G.repeat(s.G);
            elements.west.C.textContent = icons.C.repeat(s.C);
            
            elements.east.F.textContent = icons.F.repeat(1 - s.F);
            elements.east.W.textContent = icons.W.repeat(1 - s.W);
            elements.east.G.textContent = icons.G.repeat(1 - s.G);
            elements.east.C.textContent = icons.C.repeat(1 - s.C);
            
            boatP.textContent = '';
            boat.style.transform = s.F === 1 ? 'translateX(0px)' : 'translateX(200px)';
            stepLabel.textContent = 'Step ' + step + ' of __N__';
        }

        async function animateTransition(fromStep, toStep) {
            if (isAnimating) return;
            isAnimating = true;
            
            const s0 = states[fromStep];
            const s1 = states[toStep];
            
            const fPass = Math.abs(s0.F - s1.F);
            const wPass = Math.abs(s0.W - s1.W);
            const gPass = Math.abs(s0.G - s1.G);
            const cPass = Math.abs(s0.C - s1.C);
            
            if (s0.F === 1) { 
                elements.west.F.textContent = icons.F.repeat(s0.F - fPass);
                elements.west.W.textContent = icons.W.repeat(s0.W - wPass);
                elements.west.G.textContent = icons.G.repeat(s0.G - gPass);
                elements.west.C.textContent = icons.C.repeat(s0.C - cPass);
            } else { 
                elements.east.F.textContent = icons.F.repeat((1 - s0.F) - fPass);
                elements.east.W.textContent = icons.W.repeat((1 - s0.W) - wPass);
                elements.east.G.textContent = icons.G.repeat((1 - s0.G) - gPass);
                elements.east.C.textContent = icons.C.repeat((1 - s0.C) - cPass);
            }
            
            boatP.textContent = icons.F.repeat(fPass) + icons.W.repeat(wPass) + icons.G.repeat(gPass) + icons.C.repeat(cPass);
            
            const sound = playMotorSound();
            boat.style.transform = s1.F === 1 ? 'translateX(0px)' : 'translateX(200px)';
            
            await new Promise(r => setTimeout(r, 1500));
            
            stopMotorSound(sound);
            currentStep = toStep;
            renderStaticState(currentStep);
            isAnimating = false;
        }

        document.getElementById('btn-next-__UID__').addEventListener('click', () => {
            if (currentStep < states.length - 1) animateTransition(currentStep, currentStep + 1);
        });

        document.getElementById('btn-prev-__UID__').addEventListener('click', () => {
            if (currentStep > 0) animateTransition(currentStep, currentStep - 1);
        });

        document.getElementById('btn-reset-__UID__').addEventListener('click', () => {
            if (!isAnimating) {
                currentStep = 0;
                renderStaticState(0);
            }
        });

        renderStaticState(0);
    })();
    </script>
    |}

let show_animated_solution (solution : assignment) (n : int) : unit =
  let get x i = RMap.find (Printf.sprintf "%s%d" x i) solution in
  let states =
    "[" ^ String.concat ", "
            (List.map (fun i -> Printf.sprintf "{\"F\": %d, \"W\": %d, \"G\": %d, \"C\": %d}"
                                  (get "f" i) (get "w" i) (get "g" i) (get "c" i))
                      (range 0 n)) ^ "]"
  in
  let uid  = Printf.sprintf "%07x" (Random.bits ()) in
  let html = replace_all (replace_all (replace_all template "__UID__" uid) "__N__" (string_of_int n))
                         "__STATES__" states in
  ignore (Jupyter_notebook.display "text/html" html)

In [ ]:
show_animated_solution solution n